# Building and Sharing Custom Research Apps

SciTeX lets researchers create domain-specific tools (dashboards, analysis pipelines,
visualization apps) and share them via the App Store on [SciTeX Cloud](https://scitex.ai).

**Three packages work together:**
- **scitex-app** — Backend SDK: unified file storage that works identically local and on cloud
- **scitex-ui** — Frontend: shared React/TypeScript components (file browser, themes, layout)
- **scitex-cloud** — Platform: hosts, discovers, and distributes apps

## 1. Scaffold a New App

In [ ]:
# CLI: scitex template clone app my_dashboard
import scitex as stx

# Generates a complete app scaffold:
# my_dashboard/
# ├── src/my_dashboard/
# │   ├── _django/          # Django views, URLs, handlers
# │   │   ├── manifest.json # App metadata & privileges
# │   │   └── frontend/     # React TypeScript UI
# │   ├── _editor/core.py   # Core logic (framework-independent)
# │   └── _cli/gui.py       # Standalone CLI launcher
# └── pyproject.toml         # Python package config

## 2. Use the App SDK (scitex-app)

The `FilesBackend` protocol provides 7 methods (read, write, list, exists, delete, rename, copy)
that work identically on local filesystem and cloud storage.

In [ ]:
from scitex_app import get_files

# Auto-detects environment (local filesystem or cloud API)
files = get_files("./my_project")

# Same API everywhere -- local, cloud, self-hosted
files.write("data/results.csv", csv_content)
content = files.read("data/results.csv")
file_list = files.list("data/")
exists = files.exists("data/results.csv")

## 3. Frontend Components (scitex-ui)

Shared React/TypeScript components ensure visual consistency across all apps.

In [ ]:
# In your app's frontend (TypeScript/React):
#
# import { FileBrowser } from "scitex-ui/app";
# import { AppShell, ThemeProvider } from "scitex-ui/shell";
#
# Available components:
# - ThemeProvider  -- light/dark theme with semantic tokens
# - AppShell       -- workspace layout with collapsible sidebar
# - StatusBar      -- bottom bar (left/center/right sections)
# - FileBrowser    -- tree view for file navigation
# - PackageDocsSidebar -- Python package docs browser

# From Python, list available components:
from scitex_ui import list_components
list_components()

## 4. App Manifest

Apps declare their metadata and required privileges in `manifest.json`:

In [ ]:
# manifest.json (auto-generated by scitex template clone app)
manifest = {
    "name": "spike_sorter",
    "version": "1.0.0",
    "description": "Interactive spike sorting for electrophysiology data",
    "privileges": [
        {"type": "filesystem", "scope": "project"},
        {"type": "network", "scope": "outbound"},
        {"type": "api", "scope": "datastore"},
    ],
    "dependencies": {
        "python": ["numpy", "scipy", "scitex-io"],
    },
}

## 5. Publish to App Store

Apps are published to SciTeX Cloud where other researchers can browse, install,
and run them -- locally or on the cloud.

```bash
# Validate app structure
scitex-cloud app validate ./my_dashboard

# Publish to a SciTeX Cloud instance
scitex-cloud app publish ./my_dashboard --server https://scitex.ai --token $TOKEN

# Other researchers install from the App Store
# (one click in the web UI, or via CLI)
```

Apps auto-register via pip entry points -- install the package and it appears
in the workspace sidebar automatically.

## Summary

| Step | Tool | What happens |
|------|------|-------------|
| **Create** | `scitex template clone app` | Scaffold with Django views, React frontend, manifest |
| **Develop** | `scitex-app` SDK | Unified file storage (local = cloud), datastore, jobs |
| **Style** | `scitex-ui` components | Consistent themes, layout, file browser |
| **Test** | `pip install -e .` | Works identically local and on cloud |
| **Share** | `scitex-cloud app publish` | App Store on [scitex.ai](https://scitex.ai) |
| **Install** | One click in App Store | Auto-appears in workspace sidebar |